# Day 12 — Logistic regression and probability thresholds

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5-11, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column).

## 1. Understanding logistic regression

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-012/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd

# The sigmoid (logistic) function squeezes any number z into a probability between 0 and 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-6, -4, -2, 0, 2, 4, 6]:
    print(f"z = {z:>2}  ->  sigmoid(z) = {sigmoid(z):.3f}")
curve = pd.DataFrame({"z": np.arange(-6, 6.5, 0.5)})
curve["Probability"] = sigmoid(curve["z"]).round(3)
curve.to_csv("sigmoid.csv", index=False)

z = -6  ->  sigmoid(z) = 0.002
z = -4  ->  sigmoid(z) = 0.018
z = -2  ->  sigmoid(z) = 0.119
z =  0  ->  sigmoid(z) = 0.500
z =  2  ->  sigmoid(z) = 0.881
z =  4  ->  sigmoid(z) = 0.982
z =  6  ->  sigmoid(z) = 0.998


In [ ]:
# The target: at risk = did not pass (an exam score below 50). 1 = at risk, the class we want to find
df = pd.read_csv("student_data_150.csv")
df["AtRisk"] = (~df["Passed"]).astype(int)
print(df["AtRisk"].value_counts().rename({0: "0 (not at risk)", 1: "1 (at risk)"}).to_string())
print(f"Share at risk: {df['AtRisk'].mean():.1%}")

AtRisk
0 (not at risk)    106
1 (at risk)         44
Share at risk: 29.3%


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression

# One 70/15/15 split, stratified so both classes appear in every set (as on Day 9)
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["AtRisk"])
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42, stratify=train_val["AtRisk"])
print(f"Train {len(train)} | validation {len(validation)} | test {len(test)} (kept aside)")

# Why not linear regression? Fit both on study hours alone, on the training students
lin = LinearRegression().fit(train[["StudyHours"]], train["AtRisk"])
log = LogisticRegression().fit(train[["StudyHours"]], train["AtRisk"])
hours = pd.DataFrame({"StudyHours": np.arange(0.5, 7.5, 0.5)})
hours["LinearOutput"] = lin.predict(hours[["StudyHours"]]).round(3)
hours["LogisticProbability"] = log.predict_proba(hours[["StudyHours"]])[:, 1].round(3)
print(hours.iloc[[0, 4, 8, 13]].to_string(index=False))
print(f"Linear output range: {hours['LinearOutput'].min():.3f} to {hours['LinearOutput'].max():.3f}")
print(f"Logistic range:      {hours['LogisticProbability'].min():.3f} to {hours['LogisticProbability'].max():.3f}")
hours.to_csv("study_hours_curves.csv", index=False)

Train 104 | validation 23 | test 23 (kept aside)
 StudyHours  LinearOutput  LogisticProbability
        0.5         0.767                0.845
        2.5         0.457                0.457
        4.5         0.147                0.115
        7.0        -0.241                0.012
Linear output range: -0.241 to 0.767
Logistic range:      0.012 to 0.845


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Logistic regression on three features: scale, then fit on the training students
features = ["StudyHours", "Attendance", "Age"]
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(train[features], train["AtRisk"])
proba = model.predict_proba(validation[features])[:, 1]          # P(at risk) for each validation student
shown = validation[["Student"] + features + ["AtRisk"]].assign(P_at_risk=proba.round(2),
                                                               Class_at_05=(proba >= 0.5).astype(int))
print(shown.sort_values("P_at_risk", ascending=False).head(6).to_string(index=False))
pd.DataFrame({"Probability": proba.round(3),
              "Actual": np.where(validation["AtRisk"] == 1, "At risk", "Not at risk")}).to_csv(
    "validation_probabilities.csv", index=False)

Student  StudyHours  Attendance  Age  AtRisk  P_at_risk  Class_at_05
   G053         2.0        55.0 12.0       0       0.85            1
   G096         1.0        71.0 11.0       1       0.84            1
   G054         1.5        75.0 12.0       1       0.70            1
   G057         2.0        69.0 11.0       1       0.70            1
     S2         2.0        70.0 11.0       1       0.69            1
   G106         1.0        86.0 10.0       1       0.67            1


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# The same probabilities, three thresholds: each gives different class labels
y = validation["AtRisk"]
print(f"{y.sum()} of {len(y)} validation students are actually at risk")
for t in [0.3, 0.5, 0.7]:
    pred = (proba >= t).astype(int)
    tp, fp = ((pred == 1) & (y == 1)).sum(), ((pred == 1) & (y == 0)).sum()
    print(f"t = {t}: flags {pred.sum():2d} | caught {tp} of {y.sum()} | false alarms {fp} | "
          f"accuracy {accuracy_score(y, pred):.2f} | precision {precision_score(y, pred, zero_division=0):.2f} | "
          f"recall {recall_score(y, pred):.2f} | F1 {f1_score(y, pred):.2f}")
print(f"ROC-AUC (uses the probabilities, no threshold): {roc_auc_score(y, proba):.2f}")

7 of 23 validation students are actually at risk
t = 0.3: flags  8 | caught 6 of 7 | false alarms 2 | accuracy 0.87 | precision 0.75 | recall 0.86 | F1 0.80
t = 0.5: flags  6 | caught 5 of 7 | false alarms 1 | accuracy 0.87 | precision 0.83 | recall 0.71 | F1 0.77
t = 0.7: flags  4 | caught 3 of 7 | false alarms 1 | accuracy 0.78 | precision 0.75 | recall 0.43 | F1 0.55
ROC-AUC (uses the probabilities, no threshold): 0.92


## 2. Model training and interpretation

In [ ]:
# Section 2: train on the original units (no scaling, no penalty), so each coefficient is per hour, per % or per year
plain = LogisticRegression(C=np.inf, max_iter=1000).fit(train[features], train["AtRisk"])
print(f"Trained on {len(train)} students, {train['AtRisk'].sum()} of them at risk")
print(f"Intercept (b0): {plain.intercept_[0]:.3f}")
for name, b in zip(features, plain.coef_[0]):
    print(f"{name:<11} b = {b:+.3f} | odds of being at risk x {np.exp(b):.3f} per unit")

Trained on 104 students, 30 of them at risk
Intercept (b0): 8.369
StudyHours  b = -1.002 | odds of being at risk x 0.367 per unit
Attendance  b = -0.071 | odds of being at risk x 0.932 per unit
Age         b = -0.040 | odds of being at risk x 0.960 per unit


In [ ]:
# From coefficients to probability, by hand, for one real validation student
student = validation[validation["Student"] == "S2"].iloc[0]
z = plain.intercept_[0] + sum(b * student[f] for f, b in zip(features, plain.coef_[0]))
terms = " ".join(f"{'-' if b < 0 else '+'} {abs(b):.3f} x {student[f]:g}" for f, b in zip(features, plain.coef_[0]))
print(f"S2: {student['StudyHours']:g} hours, {student['Attendance']:g}% attendance, age {student['Age']:g}, actually at risk: {student['AtRisk']}")
print(f"z = {plain.intercept_[0]:.3f} {terms} = {z:.3f}")
print(f"P(at risk) = 1 / (1 + e^-z) = {1 / (1 + np.exp(-z)):.3f} | predict_proba: {plain.predict_proba(student[features].to_frame().T)[0, 1]:.3f}")
print(f"P(pass) = 1 - P(at risk) = {1 - 1 / (1 + np.exp(-z)):.3f}")

S2: 2 hours, 70% attendance, age 11, actually at risk: 1
z = 8.369 - 1.002 x 2 - 0.071 x 70 - 0.040 x 11 = 0.954
P(at risk) = 1 / (1 + e^-z) = 0.722 | predict_proba: 0.722
P(pass) = 1 - P(at risk) = 0.278


In [ ]:
# Which feature matters most? Compare coefficients on SCALED features (per standard deviation), from Section 1's model
scaled = model.named_steps["logisticregression"]
sd = train[features].std(ddof=0)
for name, b_scaled, b_unit in zip(features, scaled.coef_[0], plain.coef_[0]):
    print(f"{name:<11} per unit {b_unit:+.3f} | 1 SD = {sd[name]:5.2f} | per SD (scaled model) {b_scaled:+.3f}")

StudyHours  per unit -1.002 | 1 SD =  1.35 | per SD (scaled model) -1.214
Attendance  per unit -0.071 | 1 SD = 10.23 | per SD (scaled model) -0.659
Age         per unit -0.040 | 1 SD =  0.80 | per SD (scaled model) -0.039


In [ ]:
# The effect of study hours alone: attendance and age held at the training average
grid = pd.DataFrame({"StudyHours": np.arange(0.5, 7.5, 0.5), "Attendance": train["Attendance"].mean(), "Age": train["Age"].mean()})
grid["P_at_risk"] = plain.predict_proba(grid[features])[:, 1].round(3)
print(f"Attendance {train['Attendance'].mean():.1f}%, age {train['Age'].mean():.1f}")
print(grid.iloc[[0, 3, 5, 7, 9, 13]][["StudyHours", "P_at_risk"]].to_string(index=False))
grid[["StudyHours", "P_at_risk"]].to_csv("study_hours_effect.csv", index=False)

Attendance 80.0%, age 11.0
 StudyHours  P_at_risk
        0.5      0.852
        2.0      0.561
        3.0      0.319
        4.0      0.147
        5.0      0.059
        7.0      0.008


## 3. Probability thresholds and trade-offs

In [ ]:
# Section 3: one probability, three thresholds. Four real validation students, from Section 1's model
pick = [np.abs(proba - target).argmin() for target in [0.8, 0.6, 0.4, 0.25]]
four = validation.iloc[pick][["Student", "AtRisk"]].assign(P=proba[pick].round(2))
for t in [0.3, 0.5, 0.7]:
    four[f"t={t}"] = (proba[pick] >= t).astype(int)
print(four.to_string(index=False))

Student  AtRisk    P  t=0.3  t=0.5  t=0.7
   G096       1 0.84      1      1      1
   G106       1 0.67      1      1      0
   G123       1 0.44      1      0      0
   G013       0 0.24      0      0      0


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix

# Steadier than 23 students: out-of-fold probabilities for all 127 non-test students (5-fold, as on Day 11).
# Each student's probability comes from a model that never saw them. The test set stays aside.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), train_val[features],
                        train_val["AtRisk"], cv=folds, method="predict_proba")[:, 1]
y_tv = train_val["AtRisk"].values
print(f"{len(y_tv)} students, {y_tv.sum()} at risk")
for t in [0.3, 0.5, 0.7]:
    tn, fp, fn, tp = confusion_matrix(y_tv, (oof >= t).astype(int)).ravel()
    print(f"t = {t}: TP {tp:2d} | FP {fp:2d} | FN {fn:2d} | TN {tn:2d} | precision {tp / (tp + fp):.2f} | "
          f"recall {tp / (tp + fn):.2f} | F1 {2 * tp / (2 * tp + fp + fn):.2f}")

127 students, 37 at risk
t = 0.3: TP 27 | FP 24 | FN 10 | TN 66 | precision 0.53 | recall 0.73 | F1 0.61
t = 0.5: TP 20 | FP  8 | FN 17 | TN 82 | precision 0.71 | recall 0.54 | F1 0.62
t = 0.7: TP  8 | FP  2 | FN 29 | TN 88 | precision 0.80 | recall 0.22 | F1 0.34


In [ ]:
# Every threshold from 0.10 to 0.90: recall never rises; precision mostly rises, unevenly
rows = []
for t in np.arange(0.10, 0.91, 0.05).round(2):
    pred = (oof >= t).astype(int)
    rows.append({"Threshold": t, "Precision": round(precision_score(y_tv, pred, zero_division=0), 3),
                 "Recall": round(recall_score(y_tv, pred), 3), "F1": round(f1_score(y_tv, pred), 3)})
sweep = pd.DataFrame(rows)
print(sweep.iloc[::2].to_string(index=False))
sweep.to_csv("threshold_metrics.csv", index=False)

 Threshold  Precision  Recall    F1
       0.1      0.380   0.946 0.543
       0.2      0.493   0.892 0.635
       0.3      0.529   0.730 0.614
       0.4      0.727   0.649 0.686
       0.5      0.714   0.541 0.615
       0.6      0.778   0.378 0.509
       0.7      0.800   0.216 0.340
       0.8      0.750   0.162 0.267
       0.9      0.667   0.054 0.100

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

# Curves over ALL thresholds: ROC (true vs false positive rate) and precision-recall
fpr, tpr, _ = roc_curve(y_tv, oof)
prec, rec, _ = precision_recall_curve(y_tv, oof)
print(f"ROC-AUC {roc_auc_score(y_tv, oof):.3f} (random guessing: 0.5)")
print(f"Average precision (PR-AUC) {average_precision_score(y_tv, oof):.3f} (random guessing: {y_tv.mean():.3f}, the share at risk)")
pd.DataFrame({"FalsePositiveRate": fpr.round(3), "TruePositiveRate": tpr.round(3)}).to_csv("roc_curve.csv", index=False)
pd.DataFrame({"Recall": rec.round(3), "Precision": prec.round(3)}).to_csv("pr_curve.csv", index=False)
print(f"ROC points {len(fpr)}, PR points {len(rec)}")

ROC-AUC 0.828 (random guessing: 0.5)
Average precision (PR-AUC) 0.647 (random guessing: 0.291, the share at risk)


ROC points 37, PR points 126


In [ ]:
# Choose the threshold by the goal, fixed in advance, on these out-of-fold probabilities (never on the test set)
best_f1 = sweep.loc[sweep["F1"].idxmax()]
print(f"Goal 'balance' (highest F1):         t = {best_f1['Threshold']:.2f} | precision {best_f1['Precision']:.2f} | recall {best_f1['Recall']:.2f} | F1 {best_f1['F1']:.2f}")
catch = sweep[sweep["Recall"] >= 0.85].iloc[-1]   # the highest threshold that still catches 85% of at-risk students
print(f"Goal 'catch at least 85% at risk':   t = {catch['Threshold']:.2f} | precision {catch['Precision']:.2f} | recall {catch['Recall']:.2f} | F1 {catch['F1']:.2f}")
sure = sweep[sweep["Precision"] >= 0.8].iloc[0]    # the lowest threshold whose flags are 80% right (90% is never reached)
print(f"Goal 'flags at least 80% right':     t = {sure['Threshold']:.2f} | precision {sure['Precision']:.2f} | recall {sure['Recall']:.2f} | F1 {sure['F1']:.2f}")

Goal 'balance' (highest F1):         t = 0.40 | precision 0.73 | recall 0.65 | F1 0.69
Goal 'catch at least 85% at risk':   t = 0.20 | precision 0.49 | recall 0.89 | F1 0.64
Goal 'flags at least 80% right':     t = 0.65 | precision 0.81 | recall 0.35 | F1 0.49


## 4. Evaluation metrics

In [ ]:
# Section 4: metrics from a confusion matrix, by hand and with scikit-learn (an illustrative matrix of 200 records)
tp, fp, fn, tn = 80, 20, 10, 90
accuracy, precision, recall = (tp + tn) / (tp + fp + fn + tn), tp / (tp + fp), tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"By hand:      accuracy {accuracy:.3f} | precision {precision:.3f} | recall {recall:.3f} | F1 {f1:.3f}")
y_true = np.array([1] * tp + [0] * fp + [1] * fn + [0] * tn)          # rebuild the records behind the matrix
y_pred = np.array([1] * tp + [1] * fp + [0] * fn + [0] * tn)
print(f"scikit-learn: accuracy {accuracy_score(y_true, y_pred):.3f} | precision {precision_score(y_true, y_pred):.3f} | "
      f"recall {recall_score(y_true, y_pred):.3f} | F1 {f1_score(y_true, y_pred):.3f}")

By hand:      accuracy 0.850 | precision 0.800 | recall 0.889 | F1 0.842
scikit-learn: accuracy 0.850 | precision 0.800 | recall 0.889 | F1 0.842


In [ ]:
from sklearn.dummy import DummyClassifier

# Why accuracy alone misleads: a "model" that always predicts "not at risk", out-of-fold on the same 127 students
never = cross_val_predict(DummyClassifier(strategy="most_frequent"), train_val[features], y_tv, cv=folds)
print(f"Always 'not at risk': accuracy {accuracy_score(y_tv, never):.3f} | precision {precision_score(y_tv, never, zero_division=0):.3f}"
      f" | recall {recall_score(y_tv, never):.3f} | F1 {f1_score(y_tv, never):.3f}")
print(f"It finds {never.sum()} of the {y_tv.sum()} at-risk students")

Always 'not at risk': accuracy 0.709 | precision 0.000 | recall 0.000 | F1 0.000
It finds 0 of the 37 at-risk students


In [ ]:
# Every metric for our model at threshold 0.5, on the 127 out-of-fold probabilities from Section 3
pred_05 = (oof >= 0.5).astype(int)
print(f"Accuracy  {accuracy_score(y_tv, pred_05):.3f}")
print(f"Precision {precision_score(y_tv, pred_05):.3f}")
print(f"Recall    {recall_score(y_tv, pred_05):.3f}")
print(f"F1-score  {f1_score(y_tv, pred_05):.3f}")
print(f"ROC-AUC   {roc_auc_score(y_tv, oof):.3f}  (from the probabilities, not the labels)")
print("Confusion matrix [[TN FP] [FN TP]]:", confusion_matrix(y_tv, pred_05).tolist())

Accuracy  0.803
Precision 0.714
Recall    0.541
F1-score  0.615
ROC-AUC   0.828  (from the probabilities, not the labels)
Confusion matrix [[TN FP] [FN TP]]: [[82, 8], [17, 20]]


In [ ]:
# Compare two models fairly: same 127 students, same folds, same threshold. A: three features. B: study hours only
candidates = {"A: three features": features, "B: study hours only": ["StudyHours"]}
table = {}
for name, cols in candidates.items():
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), train_val[cols], y_tv,
                          cv=folds, method="predict_proba")[:, 1]
    lab = (p >= 0.5).astype(int)
    table[name] = {"Accuracy": accuracy_score(y_tv, lab), "Precision": precision_score(y_tv, lab),
                   "Recall": recall_score(y_tv, lab), "F1": f1_score(y_tv, lab), "ROC-AUC": roc_auc_score(y_tv, p)}
comparison = pd.DataFrame(table).round(3)
print(comparison.to_string())
comparison.rename(columns={"A: three features": "ModelA", "B: study hours only": "ModelB"}).rename_axis("Metric").reset_index().to_csv(
    "metric_comparison.csv", index=False)

           A: three features  B: study hours only
Accuracy               0.803                0.787
Precision              0.714                0.727
Recall                 0.541                0.432
F1                     0.615                0.542
ROC-AUC                0.828                0.810


## 5. End-to-end example: student risk prediction

In [ ]:
# Section 5, end to end. 1. Load and explore: who is at risk, and how do the groups differ?
df = pd.read_csv("student_data_150.csv")
df["AtRisk"] = (~df["Passed"]).astype(int)
candidates_all = ["StudyHours", "Attendance", "Age", "PreviousScore"]
print(f"{len(df)} students | at risk {df['AtRisk'].sum()} ({df['AtRisk'].mean():.1%}) | missing values {df[candidates_all].isna().sum().sum()}")
print(df.groupby("AtRisk")[candidates_all].mean().round(1).rename(index={0: "Not at risk", 1: "At risk"}).to_string())

150 students | at risk 44 (29.3%) | missing values 0


             StudyHours  Attendance   Age  PreviousScore
AtRisk                                                  
Not at risk         4.0        81.5  11.0           66.5
At risk             2.6        74.6  11.0           53.3


In [ ]:
# 2-3. The same split as Sections 1-4 (test set of 23 kept aside). Choose the features by 5-fold cross-validation
# on the 127 other students, by a rule fixed in advance: highest mean ROC-AUC
from sklearn.model_selection import cross_val_score
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["AtRisk"])
y_tv = train_val["AtRisk"].values
feature_sets = {"Three features (Sections 1-4)": ["StudyHours", "Attendance", "Age"],
                "Plus PreviousScore": ["StudyHours", "Attendance", "Age", "PreviousScore"]}
cv_auc = {}
for name, cols in feature_sets.items():
    scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), train_val[cols], y_tv, cv=folds, scoring="roc_auc")
    cv_auc[name] = scores.mean()
    print(f"{name:<30} CV ROC-AUC {scores.mean():.3f} +/- {scores.std():.3f}")
chosen_name = max(cv_auc, key=cv_auc.get)
chosen_cols = feature_sets[chosen_name]
print("Chosen:", chosen_name)

Three features (Sections 1-4)  CV ROC-AUC 0.826 +/- 0.083


Plus PreviousScore             CV ROC-AUC 0.856 +/- 0.083
Chosen: Plus PreviousScore


In [ ]:
# 4. Train the chosen model on all 127 students and interpret it (scaled: coefficients per standard deviation)
final = make_pipeline(StandardScaler(), LogisticRegression()).fit(train_val[chosen_cols], y_tv)
coefs = final.named_steps["logisticregression"].coef_[0]
print(f"Intercept {final.named_steps['logisticregression'].intercept_[0]:.2f}")
for name, b in sorted(zip(chosen_cols, coefs), key=lambda t: t[1]):
    print(f"{name:<14} coef {b:+.2f} per SD | odds ratio {np.exp(b):.2f}")

Intercept -1.53
StudyHours     coef -1.17 per SD | odds ratio 0.31
PreviousScore  coef -1.13 per SD | odds ratio 0.32
Attendance     coef -0.56 per SD | odds ratio 0.57
Age            coef -0.01 per SD | odds ratio 0.99


In [ ]:
# 5. Choose the threshold BEFORE testing: out-of-fold probabilities, goal fixed in advance = catch at least 85% of at-risk students
oof_final = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), train_val[chosen_cols], y_tv,
                              cv=folds, method="predict_proba")[:, 1]
options = []
for t in np.arange(0.10, 0.91, 0.05).round(2):
    pred = (oof_final >= t).astype(int)
    options.append((t, precision_score(y_tv, pred, zero_division=0), recall_score(y_tv, pred), f1_score(y_tv, pred)))
threshold = max(t for t, p, r, f in options if r >= 0.85)
t, p, r, f = next(o for o in options if o[0] == threshold)
print(f"Chosen threshold {threshold:.2f}: out-of-fold precision {p:.2f} | recall {r:.2f} | F1 {f:.2f}")

Chosen threshold 0.20: out-of-fold precision 0.53 | recall 0.89 | F1 0.67


In [ ]:
# 6. Test ONCE on the 23 students kept aside, at the chosen threshold (0.5 shown only for comparison)
p_test = final.predict_proba(test[chosen_cols])[:, 1]
y_test = test["AtRisk"].values
print(f"{len(y_test)} test students, {y_test.sum()} at risk")
for label, t in [("chosen", threshold), ("for comparison", 0.5)]:
    pred = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"t = {t:.2f} ({label}): accuracy {accuracy_score(y_test, pred):.2f} | precision {precision_score(y_test, pred, zero_division=0):.2f} | "
          f"recall {recall_score(y_test, pred):.2f} | F1 {f1_score(y_test, pred):.2f} | TP {tp} FP {fp} FN {fn} TN {tn}")
print(f"ROC-AUC {roc_auc_score(y_test, p_test):.3f} | average precision {average_precision_score(y_test, p_test):.3f}")
fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
pd.DataFrame({"FalsePositiveRate": fpr_t.round(3), "TruePositiveRate": tpr_t.round(3)}).to_csv("test_roc_curve.csv", index=False)

23 test students, 7 at risk
t = 0.20 (chosen): accuracy 0.70 | precision 0.50 | recall 0.71 | F1 0.59 | TP 5 FP 5 FN 2 TN 11
t = 0.50 (for comparison): accuracy 0.61 | precision 0.33 | recall 0.29 | F1 0.31 | TP 2 FP 4 FN 5 TN 12
ROC-AUC 0.714 | average precision 0.539


## 6. Mini project: student pass/fail prediction

In [ ]:
# Section 6 mini project: predict PASS (1) or fail (0). 1. Explore
df = pd.read_csv("student_data_150.csv")
df["Pass"] = df["Passed"].astype(int)
project_features = ["StudyHours", "Attendance", "PreviousScore"]      # age dropped: no effect in Sections 2 and 5
print(f"{len(df)} students | pass {df['Pass'].sum()} ({df['Pass'].mean():.1%}) | fail {(1 - df['Pass']).sum()} | missing {df[project_features].isna().sum().sum()}")
print(df.groupby("Pass")[project_features].mean().round(1).rename(index={0: "Fail", 1: "Pass"}).to_string())

150 students | pass 106 (70.7%) | fail 44 | missing 0
      StudyHours  Attendance  PreviousScore
Pass                                       
Fail         2.6        74.6           53.3
Pass         4.0        81.5           66.5


In [ ]:
# 2-5. Keep 23 test students aside (stratified on Pass, so not the same 23 as Section 5). Compare thresholds out-of-fold
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Pass"])
y_tv = train_val["Pass"].values
p_oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), train_val[project_features], y_tv,
                          cv=folds, method="predict_proba")[:, 1]
print(f"Out-of-fold on {len(y_tv)} students | ROC-AUC {roc_auc_score(y_tv, p_oof):.3f}")
for t in [0.3, 0.5, 0.7]:
    pred = (p_oof >= t).astype(int)
    print(f"t = {t}: accuracy {accuracy_score(y_tv, pred):.2f} | precision {precision_score(y_tv, pred):.2f} | "
          f"recall {recall_score(y_tv, pred):.2f} | F1 {f1_score(y_tv, pred):.2f} | wrong 'pass' {((pred == 1) & (y_tv == 0)).sum()}")

Out-of-fold on 127 students | ROC-AUC 0.849
t = 0.3: accuracy 0.80 | precision 0.78 | recall 0.99 | F1 0.87 | wrong 'pass' 25
t = 0.5: accuracy 0.80 | precision 0.84 | recall 0.88 | F1 0.86 | wrong 'pass' 15


t = 0.7: accuracy 0.78 | precision 0.90 | recall 0.78 | F1 0.83 | wrong 'pass' 8


In [ ]:
# 6. The rule, fixed in advance: a predicted pass gets no extra help, so a wrong "pass" is costly.
# Choose the lowest threshold whose out-of-fold precision for pass reaches 0.90
rows = []
for t in np.arange(0.30, 0.91, 0.05).round(2):
    pred = (p_oof >= t).astype(int)
    rows.append((t, precision_score(y_tv, pred), recall_score(y_tv, pred)))
project_t, prec_t, rec_t = next(r for r in rows if r[1] >= 0.90)
print(f"Chosen threshold {project_t:.2f}: out-of-fold precision {prec_t:.2f} | recall {rec_t:.2f}")

Chosen threshold 0.75: out-of-fold precision 0.91 | recall 0.74


In [ ]:
# 7. Train on all 127, interpret, then test ONCE
project_model = make_pipeline(StandardScaler(), LogisticRegression()).fit(train_val[project_features], y_tv)
for name, b in zip(project_features, project_model.named_steps["logisticregression"].coef_[0]):
    print(f"{name:<14} coef {b:+.2f} per SD | odds of passing x {np.exp(b):.2f}")
p_pass = project_model.predict_proba(test[project_features])[:, 1]
y_te = test["Pass"].values
pred = (p_pass >= project_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f"Test ({len(y_te)} students, {y_te.sum()} pass) at t = {project_t:.2f}: accuracy {accuracy_score(y_te, pred):.2f} | "
      f"precision {precision_score(y_te, pred):.2f} | recall {recall_score(y_te, pred):.2f} | F1 {f1_score(y_te, pred):.2f}")
print(f"TP {tp} | FP (wrong 'pass') {fp} | FN {fn} | TN {tn} | ROC-AUC {roc_auc_score(y_te, p_pass):.3f}")
fpr_p, tpr_p, _ = roc_curve(y_te, p_pass)
pd.DataFrame({"FalsePositiveRate": fpr_p.round(3), "TruePositiveRate": tpr_p.round(3)}).to_csv("project_roc_curve.csv", index=False)

StudyHours     coef +1.11 per SD | odds of passing x 3.04
Attendance     coef +0.57 per SD | odds of passing x 1.76
PreviousScore  coef +0.71 per SD | odds of passing x 2.03
Test (23 students, 16 pass) at t = 0.75: accuracy 0.83 | precision 0.93 | recall 0.81 | F1 0.87
TP 13 | FP (wrong 'pass') 1 | FN 3 | TN 6 | ROC-AUC 0.920
